# Using Mistral's MAGISTRAL to generate the Verhaal Speciaal story

** Code has been tested with Python 3.12 **

In this notebook we will create a 'Verhaal Speciaal' story.

The story will be generated by a LLM (GPT-4) based on a prompt. 

The story will be based on user input. One of the user inputs is the reading level, which is based on class/group. 

In the last part of the notebook we can evaluate the level of the generated text.

### Contents
0. Installs and imports
1. Settings and prompt
2. Generate chapter one
3. Generate chapter two & three
4. Convert story to JSON

## 0. Installs and imports

In [1]:
#!pip install mistralai --upgrade

In [2]:
!which python3

/opt/anaconda3/bin/python3


In [3]:
import mistralai
mistralai.__version__

'1.8.2'

In [4]:
#import the local files
import config
import leesniveaus

## 1. Settings

### Setting the reading levels
Four different reading levels have been defined, see below.

Both characters have their own reading level as Verhaal Speciaal is meant to be a reading combination for parent and child. Example: parent can have reading level 3, while the child can have reading level 1.  All combinations are possible. 

### User input

The user_input prompt collects the input the user of the story creates. This is taken from the javascript code of the original Verhaal Speciaal:
1. personage een        => character_one
2. personage twee       => character_two
3. wat                  => plot
4. waarom               => reasoning
5. waar                 => setting
6. wanneer              => time

And we set the reading levels:
7. klas/groep           => groep (will be mapped to reading_level)
8. Leesniveau ouder     => reading level

### Building the prompt 

The reading levels and user input are then used to build up a prompt..

We will generate a prompt consisting of three 'sub-prompts':

prompt =  basic_prompt + previous_text + chapter_prompt 

**basic_prompt**

The basic prompt sets the structure of the story. It defines there are two characters and a story teller.  It makes sure the story follows a pattern.  

**previous_text**

Only used for chapters 2 and 3. 
Input of the previous chapter(s): one or two. 

**chapter_prompt**

This are the chapter specific inputs:

1. Start new story, introduce characters, leave room for chapters 2 and 3
2. Follow up on chapter 1, use previous text and leave room for chapter 3
3. chapter 3: Final chapter, end the story, use previous text.

### Reading levels

In [5]:
#variables based on the reading level settings
level_one = leesniveaus.level_one
level_two = leesniveaus.level_two
level_three = leesniveaus.level_three
level_four = leesniveaus.level_four


### User input

In [6]:
#These are the variables from the front end about the story . 
character_one = 'henk'
character_two = 'piet'
plot ='een wandeling'
reasoning = 'ze verdwalen'
setting = 'in het bos' #waar
time = 'in de zomer'

# These are the input variables from the front end for the reading level
group_child = 3 #class the child is in 3,4,5,6,7,8
reading_level_parent = level_one # 1 2 3 4 based on reading level settings 

In [7]:
#Reading level conversion table CHILD

group = group_child #class the child is in 3,4,5,6,7,8

if group < 4:
    reading_level_child = level_one
    print(reading_level_child)
elif group == 4:
    reading_level_child = level_two
    print(reading_level_child)
elif group <= 6:
    reading_level_child = level_three
    print(reading_level_child)
elif group <= 8:
    reading_level_child = level_four
    print(reading_level_child)


leesniveau 1: 
tekst bestaat uit korte woorden die je precies zo schrijft zoals je ze uitspreekt. voorbeelden hiervan zijn maan, bos, man, roos. 
er mogen dus geen woorden voorkomen met bijvoorbeeld sch- en -ng en -nk, -b, -d(t), -ch(t), -ooi, -aai, -oei, -eeuw, -ieuw, -uw. 
de zinnen zijn zo kort mogelijk. 
elke zin begint op een nieuwe regel. 
er komen geen in hoofdletters voor, dus alle woorden worden met kleine letters geschreven. 
dit geldt ook voor de titel en de namen van de personen
een voorbeeld hoe je een naam schrijft: eddy, jan en marieke.



In [8]:
#summarizing the reading levels
print(f'Reading level child: {reading_level_child[11:12]}')
print(f'Reading level parent: {reading_level_parent[11:12]}')

Reading level child: 1
Reading level parent: 1


### basic_prompt

In [9]:
#update reading levels
basic_prompt_v4 =f'''Je bent een kinderboekenschrijver. 
Je schrijft een verhaal het Nederlands waarbij je de drie-hoofdstukken-structuur van een toneelstuk volgt.
Dit is een scriptdialoog tussen twee personages en er is een Verteller die de scène schetst. 

Het leesniveau van de verteller is {reading_level_parent}.

Er zijn twee karakers die ieder een eigen leesniveau hebben. Hierna volgen de regels per niveau. Daarna wordt aangegeven welk niveau ieder personage heeft. 
Dit is een beschrijving van personage {character_one}.
Het leesniveau van personage {character_one} is niveau {reading_level_child}, dus houd het taalgebruik op dat niveau voor dit personage. Gebruik hiervoor de omschrijving van de hiervoor genoemde niveuas
Dit is een beschrijving van personage {character_two}.
Het leesniveau van personage {character_two} is niveau {reading_level_parent}, dus houd het taalgebruik op dat niveau voor dit personage. Gebruik hiervoor de omschrijving van de hiervoor genoemde niveaus houdt het hoofdstuk bij twee zinnen per karakter.

De algemene verhaallijn is: {plot}.
Dit is de reden achter het verhaal: {reasoning}.
De setting van het verhaal is: {setting}.
De tijd waarin het verhaal zich afspeelt is: {time}.

Gebruik de volgende regels om te output te structureren:
Iedere zin of paragraaf van het verhaal moet bij de Verteller, {character_one} of {character_two} horen. 
De verteller wordt altijd aangeduid als Verteller. Gebruik het format Verteller | tekst
Voeg geen code tussen haakjes toe voor de Verteller.

Als een personage wat gaat vertellen, voeg {{char1}} of {{char2}} toe voor de naam van het personage dat spreekt.
voeg een | tussen alle woorden in zoals in dit voorbeeld: {{char1}} | {character_one} | tekst.
Aan het einde van het hoofdstuk moet de verteller een vraag stellen aan een van de personages over de voorgaande dialoog.
Aan het einde van het hoofdstuk moet de tekst '''"{ENDOFACT}"''' op een nieuwe regel worden toegevoegd.
Begin het hoofdstuk duidelijk met het nummer van het hoofdstuk. Bijvoorbeeld: 'Hoofdstuk 1'.
Zorg ervoor dat de personages hetzelfde blijven in de verschillende hoofdstukken en dat ze weten wat er gezegd is.
Voeg geen uitleg toe, alleen de dialoog.
Voeg geen nieuwe personages of settings toe aan de dialoog.
De allereerste regel van de tekst moet een gegenereerde titel zijn. Gebruik alleen letters en spaties, in de titel staat niet het woord 'titel'.
Gebruik geen speciale tekens in de tekst, alleen letters, spaties en nieuwe regels.
'''

### Previous text

In [10]:
#for chapter one empty, for chapter 2/3 will be updated, see below
previous = " " 

### Chapter prompts

In [11]:
chapter_1 = f'''Dit is het eerste hoofdstuk van drie, zorg dus dat het verhaal verder kan gaan.'''
chapter_2 = f'''Dit is het tweede hoofdstuk van drie. Ga door op het eerste hoofdstuk wat je uit deze tekst haalt: {previous}. Zorg dat het verhaal verder kan gaan in hoofdstuk 3. Begin de tekst met de titel van het verhaal en dan Hoofdstuk 2 '''
chapter_3 = f'''Dit is het laatste hoofdstuk dus zorg voor een goed en happy einde. Ga door met het verhaal gebaseerd op hoofdstuk 1 en 2 wat je uit de deze tekst haalt: {previous}. Begin de tekst met de titel van het verhaal en dan Hoofdstuk 3.'''

### Concatenate prompt
We will use v3 as this is the 3rd version of the prompt (to keep it similar to original Verhaal Speciaal)

prompt_v3 = basic_prompt + chapter_prompt

In [12]:
prompt_v4_ch1 = basic_prompt_v4 + chapter_1 
print(prompt_v4_ch1)

Je bent een kinderboekenschrijver. 
Je schrijft een verhaal het Nederlands waarbij je de drie-hoofdstukken-structuur van een toneelstuk volgt.
Dit is een scriptdialoog tussen twee personages en er is een Verteller die de scène schetst. 

Het leesniveau van de verteller is leesniveau 1: 
tekst bestaat uit korte woorden die je precies zo schrijft zoals je ze uitspreekt. voorbeelden hiervan zijn maan, bos, man, roos. 
er mogen dus geen woorden voorkomen met bijvoorbeeld sch- en -ng en -nk, -b, -d(t), -ch(t), -ooi, -aai, -oei, -eeuw, -ieuw, -uw. 
de zinnen zijn zo kort mogelijk. 
elke zin begint op een nieuwe regel. 
er komen geen in hoofdletters voor, dus alle woorden worden met kleine letters geschreven. 
dit geldt ook voor de titel en de namen van de personen
een voorbeeld hoe je een naam schrijft: eddy, jan en marieke.
.

Er zijn twee karakers die ieder een eigen leesniveau hebben. Hierna volgen de regels per niveau. Daarna wordt aangegeven welk niveau ieder personage heeft. 
Dit is ee

## 2. Generate chapter one with Mistral

In [24]:
import config
api_key = config.MISTRAL_API_KEY


In [ ]:

from mistralai import Mistral

def create_chat_completion(prompt, model= 'mistral-large-2402'): 
    """
    Creates a chat completion using the Mistral API
    """
    try:
        # Initialize Mistral client
        client = Mistral(api_key=config.MISTRAL_API_KEY)
        
        # Create chat completion
        response = client.chat.complete(
            model=model,
            messages=[{
                "role": "user",
                "content": prompt
            }]
        )
        
        # Extract and return the response text
        return response.choices[0].message.content
        
    except Exception as e:
        print(f"Error creating chat completion: {e}")
        return None

# Example usage:
chapter_one = create_chat_completion(prompt_v4_ch1)

print(chapter_one)

de wandeling

hoofdstuk 1

verteller | het is een mooie dag.
verteller | henk en piet gaan wandelen.
verteller | ze lopen door het bos.
verteller | het is zomer.

{char1} | henk | wat een mooie dag.
{char1} | henk | het is warm.
{char2} | piet | ja, het is zomer.
{char2} | piet | ik hou van zomer.

verteller | henk en piet lopen verder.
verteller | ze zien veel bomen.
verteller | ze horen vogels zingen.
verteller | ze lopen een pad op.

{char1} | henk | waar gaat dit pad heen?
{char2} | piet | ik weet het niet.
{char2} | piet | laten we kijken.
{char1} | henk | ok, laten we kijken.

verteller | henk en piet lopen verder.
verteller | ze komen bij een bocht.
verteller | ze zien een klein pad.
verteller | wat doen ze nu?

{ENDOFACT}


In [15]:
chapter_one = create_chat_completion(prompt_v4_ch1)
print(chapter_one)

de avontuur van henk en piet

hoofdstuk 1

verteller | het is een warme zomer dag
verteller | henk en piet gaan een wandeling maken in het bos

{char1} | henk | wat een mooie dag
{char1} | henk | het is warm

{char2} | piet | ja
{char2} | piet | ik houd van de zon

verteller | henk en piet lopen verder het bos in
verteller | ze zien veel bomen en bloemen

{char1} | henk | wat een groot bos
{char1} | henk | waar is de weg

verteller | piet ziet henk aan en lacht

{char2} | piet | we zullen wel terug vinden
{char2} | piet | we hebben ons nog nooit verloopt

verteller | terwijl ze verder lopen, verdwalen ze steeds dieper in het bos
verteller | wat zullen henk en piet doen

{ENDOFACT}

hoofdstuk 2

verteller | henk en piet lopen steeds verder het bos in
verteller | ze komen aan een kruispunt

{char1} | henk | waar gaan we heen
{char1} | henk | ik weet het niet

{char2} | piet | laat ons naar links gaan
{char2} | piet | daar zien we zonlicht

verteller | ze gaan naar links en komen aan een 

## 3. Generate chapter two & three

The first chapter is input for chapter two and three. 



In [16]:
previous = f"Het vorige hoofdstuk was: {chapter_one}."

In [17]:
prompt_v4_ch2 = basic_prompt_v4+previous+chapter_2
print(prompt_v4_ch2)

Je bent een kinderboekenschrijver. 
Je schrijft een verhaal het Nederlands waarbij je de drie-hoofdstukken-structuur van een toneelstuk volgt.
Dit is een scriptdialoog tussen twee personages en er is een Verteller die de scène schetst. 

Het leesniveau van de verteller is leesniveau 1: 
tekst bestaat uit korte woorden die je precies zo schrijft zoals je ze uitspreekt. voorbeelden hiervan zijn maan, bos, man, roos. 
er mogen dus geen woorden voorkomen met bijvoorbeeld sch- en -ng en -nk, -b, -d(t), -ch(t), -ooi, -aai, -oei, -eeuw, -ieuw, -uw. 
de zinnen zijn zo kort mogelijk. 
elke zin begint op een nieuwe regel. 
er komen geen in hoofdletters voor, dus alle woorden worden met kleine letters geschreven. 
dit geldt ook voor de titel en de namen van de personen
een voorbeeld hoe je een naam schrijft: eddy, jan en marieke.
.

Er zijn twee karakers die ieder een eigen leesniveau hebben. Hierna volgen de regels per niveau. Daarna wordt aangegeven welk niveau ieder personage heeft. 
Dit is ee

In [18]:
chapter_two = create_chat_completion(prompt_v4_ch2)
print(chapter_two)

de avontuur van henk en piet

hoofdstuk 2

verteller | henk en piet lopen steeds verder het bos in
verteller | ze komen aan een kruispunt

{char1} | henk | waar gaan we heen
{char1} | henk | ik weet het niet

{char2} | piet | laat ons naar links gaan
{char2} | piet | daar zien we zonlicht

verteller | ze gaan naar links en komen aan een rivier
verteller | de rivier is breed en diep

{char1} | henk | hoe komen we over
{char1} | henk | ik kan niet zwemmen

{char2} | piet | laat ons een brug zoeken
{char2} | piet | er moet een weg zijn

verteller | henk en piet lopen langs de rivier
verteller | ze zoeken naar een brug

verteller | wat zal henk en piet vinden op hun zoektocht

{ENDOFACT}


In [19]:
#calling chapter three
previous = f"De vorige hoofdstukken waren {chapter_one} en {chapter_two}."
prompt_v4_ch3 = basic_prompt_v4+previous+chapter_3
print(prompt_v4_ch3)

Je bent een kinderboekenschrijver. 
Je schrijft een verhaal het Nederlands waarbij je de drie-hoofdstukken-structuur van een toneelstuk volgt.
Dit is een scriptdialoog tussen twee personages en er is een Verteller die de scène schetst. 

Het leesniveau van de verteller is leesniveau 1: 
tekst bestaat uit korte woorden die je precies zo schrijft zoals je ze uitspreekt. voorbeelden hiervan zijn maan, bos, man, roos. 
er mogen dus geen woorden voorkomen met bijvoorbeeld sch- en -ng en -nk, -b, -d(t), -ch(t), -ooi, -aai, -oei, -eeuw, -ieuw, -uw. 
de zinnen zijn zo kort mogelijk. 
elke zin begint op een nieuwe regel. 
er komen geen in hoofdletters voor, dus alle woorden worden met kleine letters geschreven. 
dit geldt ook voor de titel en de namen van de personen
een voorbeeld hoe je een naam schrijft: eddy, jan en marieke.
.

Er zijn twee karakers die ieder een eigen leesniveau hebben. Hierna volgen de regels per niveau. Daarna wordt aangegeven welk niveau ieder personage heeft. 
Dit is ee

In [20]:
chapter_three = create_chat_completion(prompt_v4_ch3)
print(chapter_three)

het avontuur van henk en piet

hoofdstuk 3

verteller | henk en piet lopen langs de rivier
verteller | ze zoeken naar een brug

{char1} | henk | ik hoop dat we snel een brug vinden
{char1} | henk | ik ben moe

{char2} | piet | ja
{char2} | piet | maar we moeten doorzetten

verteller | na een tijdje zien ze een brug
verteller | ze zijn blij en gaan ernaartoe

{char1} | henk | we hebben het gedaan
{char1} | henk | we zijn over de rivier

{char2} | piet | ja
{char2} | piet | maar waar zijn we nu

verteller | ze zien een licht in de verte
verteller | ze gaan ernaartoe

{char1} | henk | ik zie een huis
{char1} | henk | misschien kunnen ze ons helpen

{char2} | piet | ja
{char2} | piet | laat ons gaan

verteller | henk en piet kloppen aan de deur
verteller | een vriendelijke man opent de deur

{char1} | henk | hallo
{char1} | henk | we zijn verloopt

{char2} | piet | kun je ons helpen
{char2} | piet | we weten niet waar we zijn

verteller | de man lacht en zegt
verteller | kom binnen, ik hel

## 4. Save story as JSON

We need to create a .json file out of this story. This .json file will be analysed by the other scripts in the 'AVI Score' repository. 

The JSON structure is relatively simple, containing just one key-value pair. The complexity lies in the structured text content rather than in nested JSON objects or arrays.

The JSON file contains a single object with one key-value pair:
Key: "text"
Value: A long string containing a story

- First line contains the title
- Narrator sections: Paragraphs starting with "Verteller |"
- Character dialogues: Lines starting with "{char1} |" or "{char2} |"

Character dialogues follow this pattern:
- {char1} | anna | [dialogue text]
- {char2} | tom | [dialogue text]

Other things
- "{ENDOFACT}" appears at the end, likely indicating the end of a story act or section.
- The text uses newline characters (\n) to separate lines and sections.
- There are no nested objects or arrays within this JSON structure.
- The entire story is contained within a single string value.


In [21]:
story = {"text": chapter_one + chapter_two + chapter_three}
print(story['text'])

de avontuur van henk en piet

hoofdstuk 1

verteller | het is een warme zomer dag
verteller | henk en piet gaan een wandeling maken in het bos

{char1} | henk | wat een mooie dag
{char1} | henk | het is warm

{char2} | piet | ja
{char2} | piet | ik houd van de zon

verteller | henk en piet lopen verder het bos in
verteller | ze zien veel bomen en bloemen

{char1} | henk | wat een groot bos
{char1} | henk | waar is de weg

verteller | piet ziet henk aan en lacht

{char2} | piet | we zullen wel terug vinden
{char2} | piet | we hebben ons nog nooit verloopt

verteller | terwijl ze verder lopen, verdwalen ze steeds dieper in het bos
verteller | wat zullen henk en piet doen

{ENDOFACT}

hoofdstuk 2

verteller | henk en piet lopen steeds verder het bos in
verteller | ze komen aan een kruispunt

{char1} | henk | waar gaan we heen
{char1} | henk | ik weet het niet

{char2} | piet | laat ons naar links gaan
{char2} | piet | daar zien we zonlicht

verteller | ze gaan naar links en komen aan een 

In [22]:
import json
import os
from datetime import datetime

# Convert the string into a JSON serializable format, e.g., as a dictionary
story_to_save = story

# Get the current date
current_date = datetime.now().strftime("%Y-%m-%d_%H:%M")

# Create a filename with the current date
filename = f"vs_mistral_rl{reading_level_child[11:12]}_{current_date}.json"
file_path = os.path.join('json', filename)

# Save the data to a JSON file
with open(file_path, 'w') as json_file:
    json.dump(story_to_save, json_file, indent=4)

print(f"Data saved to {filename}")

Data saved to vs_mistral_rl1_2025-06-19_16:38.json


## 5. Validate the new .json with an old example

In [23]:
#Here we use an old .json based on the Javascript code base

import json
from pprint import pprint

# Read the JSON file
with open('./json/V_S_2025-05-27_12:20.json', 'r') as file:
    data = json.load(file)

# Pretty print using json.dumps()
print("Pretty printed using json.dumps():")
print(json.dumps(data, indent=4))

# Pretty print using pprint
print("\nPretty printed using pprint:")
pprint(data['text'])

FileNotFoundError: [Errno 2] No such file or directory: './json/V_S_2025-05-27_12:20.json'

## To do list
- take prompts to config files instead of code

 
